# Lab 3.1 — Configuring Claude Code
### CLAUDE.md Hierarchy, Commands & Skills — Reflection & Self-Check Answers

## Exercise 1 — CLAUDE.md hierarchy and @import

**Q1. Why keep rules in small @imported files instead of pasting everything into one big CLAUDE.md — what do you gain in maintenance and reuse?**

Each file has one concern (style vs. testing), so it is obvious where a new rule belongs and PR diffs stay small and reviewable. It also enables reuse: `style.md` or `testing.md` could be imported by another related service without duplicating text. A single giant CLAUDE.md becomes a dumping ground that is hard to scan and easy for people to skip past.

**Q2. Claude answered the testing-rules question without opening testing.md. What does that tell you about when project memory is loaded, and why does that matter for every later request?**

Project memory, the root CLAUDE.md and everything it @imports, is loaded once at session startup rather than fetched lazily per question. That means every request already has the team's conventions in context for free, with no need to re-explain them each time. The flip side: edits to CLAUDE.md or a rules file made mid-session will not take effect until the session restarts, which is exactly why Exercise 1 Step 3 requires a restart to pick up the new user-level rule.

**Q3. A user-level ~/.claude/CLAUDE.md rule and a project rule could conflict. How does the hierarchy resolve that, and when would you put a rule at the user level vs. the project level?**

The more specific level wins on conflict: project-level (and its imported modules) takes precedence over user-level, since project conventions are what this codebase specifically needs. Put a rule at the user level when it is a personal habit you want everywhere regardless of project (e.g. explain a change in one sentence before editing). Put a rule at the project level when it is a team convention specific to this codebase (e.g. money is float USD, rounded at the boundaries).

## Exercise 2 — Slash commands

**Q1. The /review command lists read-only allowed-tools and says do not edit files. Why scope a command's tools so tightly — what does least privilege buy you here?**

Restricting allowed-tools to `git diff`, `git status`, `Read`, and `Grep` means /review is physically incapable of changing code, no matter how the prompt is worded or how a future edit to the prompt text drifts. Least privilege turns "please don't edit files" from a soft instruction the model could ignore or misinterpret into a hard guarantee enforced by the harness.

**Q2. /test and /review are just Markdown files in the repo. What do you gain by checking them in versus each person typing the prompt by hand every time?**

Everyone runs the identical, versioned procedure instead of each person's slightly different half-remembered version of "go run the tests." Changes to the checklist go through code review like any other change, and onboarding a new teammate is "look in .claude/commands/" instead of tribal knowledge passed around informally.

**Q3. What is $ARGUMENTS for in review.md, and how does it let one command serve many situations?**

`$ARGUMENTS` is substituted with whatever text follows the command, e.g. `pricing.py` in `/review pricing.py`. It lets one generic command file serve many different scopes: a single file, a directory, or the whole diff with no argument, without needing a separate command per target.

## Exercise 3 — Skills

**Q1. A skill is auto-invoked by its description; a slash command is called explicitly by name. When is each the right way to package a piece of work?**

Use a slash command for a deliberate, on-demand chore the user consciously decides to trigger ("run the tests now"). Use a skill when the trigger is really about intent expressed in plain language and the user shouldn't have to remember a specific name; "update the changelog" should work however it's phrased, because it's a workflow the agent should recognize on its own.

**Q2. Why does the quality of the description field matter so much for a skill — what happens if it is too narrow, or too broad?**

The description is the entire matching mechanism. Too narrow, and the skill silently never fires: a user phrases the request slightly differently and gets generic behavior instead of the packaged workflow. Too broad, and the skill hijacks unrelated requests that merely share vocabulary. Its precision directly determines whether the skill helps or gets in the way.

**Q3. The SKILL.md bakes in judgment ("user-facing sentences," "skip formatting-only edits"). Why encode that in the skill rather than leaving it to each run — how does this connect to why rules live in CLAUDE.md?**

Baking the judgment call into the skill means every invocation applies the same standard automatically, instead of depending on whoever is typing the prompt that day to remember and restate it. It is the same reasoning as CLAUDE.md: encode the team's judgment once, in a versioned file, so it is applied consistently and improvable by PR. A skill just packages that judgment for a specific multi-step task rather than as a standing rule.

## 5.1 Self-Check Before You Leave

**1. Where does project memory live, how does @import work, and what are the three levels of the memory hierarchy?**

Project memory is `./CLAUDE.md`, loaded automatically at startup. `@path` inlines another file's content into memory at that point. Three levels, most specific wins: user (`~/.claude/CLAUDE.md`, every project), project (`./CLAUDE.md`), and imported modules (`.claude/rules/*.md`).

**2. How is a slash command defined, and what determines its name? Name the three frontmatter fields used in this lab.**

A slash command is a Markdown file in `.claude/commands/<name>.md`; the filename is the command name. Frontmatter fields used: `description`, `allowed-tools`, `argument-hint`.

**3. Why scope a command's allowed-tools, and what does $ARGUMENTS do?**

Scoping enforces least privilege: a review command literally cannot edit files. `$ARGUMENTS` injects whatever text the user typed after the command, letting one file serve many targets.

**4. What makes a skill fire? How is that different from invoking a slash command?**

A skill fires when a request's wording matches its `description`; it is auto-invoked, never called by name. A slash command only runs when the user types its exact name.

**5. Across all three, what is the common payoff of putting configuration in the repo?**

Tacit conventions become versioned, consistent across the team, reviewable via pull request, and improvable in one place instead of living in each person's head.